In [ ]:
!pip install faiss-cpu
!pip install umap-learn



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 15.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import faiss
import json
import numpy as np
import umap
import pandas as pd
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score

# Load FAISS index
faiss_index = faiss.read_index("/content/drive/MyDrive/Embeddings/faiss_index_QWEN_V5.idx")
print(f"Index loaded with {faiss_index.ntotal} vectors")

# Load metadata
with open("/content/drive/MyDrive/Embeddings/metadata_store_QWEN_V5.json", "r") as f:
    metadata_store = json.load(f)
print(f"Metadata loaded with {len(metadata_store)} entries")

# Enable FAISS reconstruction
faiss_index.make_direct_map()
num_vectors = faiss_index.ntotal
embedding_dim = faiss_index.d

# Extract stored embeddings
stored_vectors = np.zeros((num_vectors, embedding_dim), dtype=np.float32)
for i in range(num_vectors):
    stored_vectors[i] = faiss_index.reconstruct(i)

# Perform UMAP dimensionality reduction
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
embeddings_2d = umap_model.fit_transform(stored_vectors)

# Convert metadata to DataFrame
df = pd.DataFrame(metadata_store)

# Rename columns
df = df.rename(columns={'project_type': 'Bank', 'sector': 'Sector', 'data_id': 'Doc_ID', 'section': 'Section'})

# Convert necessary columns to strings
df['Sector'] = df['Sector'].astype(str)
df['Section'] = df['Section'].astype(str)

# Add UMAP coordinates to DataFrame
df['UMAP1'] = embeddings_2d[:, 0]
df['UMAP2'] = embeddings_2d[:, 1]

print("Shape of embeddings:", stored_vectors.shape)
print("Shape of UMAP embeddings:", embeddings_2d.shape)
print("Number of rows in DataFrame:", len(df))

# Perform KMeans clustering for NMI scoring
num_clusters = df['Sector'].nunique()
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(embeddings_2d)

# Calculate NMI Score
true_labels = df['Sector'].astype("category").cat.codes
predicted_labels = df['Cluster']
nmi_score = normalized_mutual_info_score(true_labels, predicted_labels)

print(f"\n✅ NMI Score for Clustering: {nmi_score:.4f}")

# Define Colors and Shapes
BANK_SHAPES = {
    'World Bank': 'circle',
    'AFDB': 'diamond',
    'AIIB': 'square',
    'IDB': 'triangle-up',
    'ADB': 'star'
}

SECTOR_COLORS = {
    'ENERGY AND EXTRACTIVES': 'rgba(255, 99, 132, 0.7)',
    'TRANSPORTATION': 'rgba(54, 162, 235, 0.7)',
    'WATER, SANITATION AND WASTE MANAGEMENT': 'rgba(85, 192, 192, 0.7)',
    'FINANCIAL SECTOR': 'rgba(153, 102, 255, 0.7)',
    'Finance': 'rgba(255, 159, 64, 0.7)',
    'HEALTH': 'rgba(255, 205, 86, 0.7)',
    'MULTI-SECTOR': 'rgba(201, 203, 207, 0.7)',
    'AGRICULTURE, FISHING AND FORESTRY': 'rgba(102, 205, 170, 0.7)',
    'INFORMATION AND COMMUNICATIONS TECHNOLOGIES': 'rgba(255, 99, 71, 0.7)',
    'SOCIAL SUPPORT': 'rgba(50, 150, 235, 0.7)',
    'INDUSTRY, TRADE AND SERVICES': 'rgba(255, 179, 64, 0.7)',
    'PUBLIC ADMINISTRATION': 'rgba(85, 192, 192, 0.7)',
    'EDUCATION': 'rgba(169, 169, 169, 0.7)',
    'Other': 'rgba(255, 255, 255, 0.7)'
}

# Create interactive Plotly scatter plot
fig = go.Figure()
traces_info = []

for bank in df['Bank'].unique():
    for sector in df['Sector'].unique():
        for section in df['Section'].unique():
            mask = (df['Bank'] == bank) & (df['Sector'] == sector) & (df['Section'] == section)
            if any(mask):
                fig.add_trace(
                    go.Scatter(
                        x=df[mask]['UMAP1'],
                        y=df[mask]['UMAP2'],
                        mode='markers',
                        name=f"{bank} - {sector} ({section})",
                        marker=dict(
                            size=6,
                            symbol=BANK_SHAPES.get(bank, 'circle'),
                            color=SECTOR_COLORS.get(sector, 'rgba(169, 169, 169, 0.7)'),
                            opacity=0.7,
                            line=dict(width=1, color='rgba(0, 0, 0, 0.5)')
                        ),
                        customdata=df[mask][['Bank', 'Sector', 'Section', 'Doc_ID']].values,
                        hovertemplate="<b>Bank:</b> %{customdata[0]}<br>"
                                      "<b>Sector:</b> %{customdata[1]}<br>"
                                      "<b>Section:</b> %{customdata[2]}<br>"
                                      "<b>Document ID:</b> %{customdata[3]}<br>"
                                      "<extra></extra>"
                    )
                )
                traces_info.append({'bank': bank, 'sector': sector, 'section': section})

# Create Bank selection buttons
bank_buttons = [
    dict(label="All Banks", method="update", args=[{"visible": [True] * len(traces_info)}])
]
for bank in sorted(df['Bank'].unique()):
    visibility = [trace['bank'] == bank for trace in traces_info]
    bank_buttons.append(dict(label=bank, method="update", args=[{"visible": visibility}]))

# Create Section selection buttons
section_buttons = [
    dict(label="All Sections", method="update", args=[{"visible": [True] * len(traces_info)}])
]
for section in sorted(df['Section'].unique()):
    visibility = [trace['section'] == section for trace in traces_info]
    section_buttons.append(dict(label=section, method="update", args=[{"visible": visibility}]))

# Create Sector selection buttons
sector_buttons = [
    dict(label="All Sectors", method="update", args=[{"visible": [True] * len(traces_info)}])
]
for sector in sorted(df['Sector'].unique()):
    visibility = [trace['sector'] == sector for trace in traces_info]
    sector_buttons.append(dict(label=sector, method="update", args=[{"visible": visibility}]))

# Update layout with dropdown menus
fig.update_layout(
    updatemenus=[
        dict(buttons=bank_buttons, direction="down", showactive=True, x=0.1, y=1.1, xanchor="left", yanchor="top", name="Bank"),
        dict(buttons=section_buttons, direction="down", showactive=True, x=0.4, y=1.1, xanchor="left", yanchor="top", name="Section"),
        dict(buttons=sector_buttons, direction="down", showactive=True, x=0.7, y=1.1, xanchor="left", yanchor="top", name="Sector"),
    ],
    title="Development Bank Embeddings (UMAP Projection)",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    showlegend=True
)

# Show the interactive plot
fig.show()


Index loaded with 8793 vectors
Metadata loaded with 8793 entries


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Shape of embeddings: (8793, 3584)
Shape of UMAP embeddings: (8793, 2)
Number of rows in DataFrame: 8793

✅ NMI Score for Clustering: 0.5032


In [ ]:
# Compute individual NMI scores for each bank
bank_nmi_scores = {}

for bank in df['Bank'].unique():
    df_bank = df[df['Bank'] == bank]  # Filter data for this bank

    if df_bank['Sector'].nunique() > 1:  # Ensure there is more than one sector for clustering
        num_clusters_bank = df_bank['Sector'].nunique()
        kmeans_bank = KMeans(n_clusters=num_clusters_bank, random_state=42, n_init=10)
        df_bank['Cluster'] = kmeans_bank.fit_predict(df_bank[['UMAP1', 'UMAP2']])

        true_labels_bank = df_bank['Sector'].astype("category").cat.codes
        predicted_labels_bank = df_bank['Cluster']

        nmi_score_bank = normalized_mutual_info_score(true_labels_bank, predicted_labels_bank)
        bank_nmi_scores[bank] = nmi_score_bank
    else:
        bank_nmi_scores[bank] = None  # NMI not applicable for single-sector banks

# Print individual bank NMI scores
print("\n✅ Individual NMI Scores by Bank:")
for bank, score in bank_nmi_scores.items():
    if score is not None:
        print(f"  - {bank}: {score:.4f}")
    else:
        print(f"  - {bank}: N/A (only one sector present)")


<ipython-input-5-f66a1f4c5631>:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-5-f66a1f4c5631>:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-5-f66a1f4c5631>:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

<ipython-input-5-f66a1f4c


✅ Individual NMI Scores by Bank:
  - ADB: 0.4598
  - AIIB: 0.3894
  - AFDB: 0.6163
  - IDB: 0.5595
  - World Bank: 0.5496


<ipython-input-5-f66a1f4c5631>:10: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

